In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

C:\Users\ANAMIKA\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
dataset=pd.read_csv('Churn_Modelling.csv')

In [4]:
dataset.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
x=dataset.iloc[:,3:13]
y=dataset['Exited']

In [6]:
x.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,699,France,Female,39,1,0.00,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10


In [7]:
#encoding
geography =pd.get_dummies(x['Geography'],drop_first=True)
gender =pd.get_dummies(x['Gender'],drop_first=True)

In [8]:
x=x.drop(['Geography','Gender'],axis=1)
x.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,42,2,0.00,1,1,1,101348.88
1,608,41,1,83807.86,1,0,1,112542.58
2,502,42,8,159660.80,3,1,0,113931.57
3,699,39,1,0.00,2,0,0,93826.63
4,850,43,2,125510.82,1,1,1,79084.10


In [10]:
x=pd.concat([x,geography,gender],axis=1)

In [11]:
x.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Germany,Spain,Male
0,619,42,2,0.00,1,1,1,101348.88,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,False,True,False


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(x, y, test_size=0.33, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [14]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [16]:
X_train.shape

(6700, 11)

In [17]:
model = Sequential([
    Dense(64, input_dim=X_train.shape[1], activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

C:\Users\ANAMIKA\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint,ReduceLROnPlateau

In [19]:
# It is a callback that reduces the learning rate (LR) when the model stops improving.
# Only treat a change in the monitored metric as improvement if it is greater than 0.0001

In [21]:
# 4) Callbacks
early_stopping = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,
    patience=10,
    verbose=1,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint("best_model.h5", monitor="val_loss", save_best_only=True, verbose=1)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

callbacks = [early_stopping, checkpoint, reduce_lr]

In [22]:
# 5) Fit
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    shuffle=True,  # It shuffles the training samples before each epoch.
    verbose=2
)

Epoch 1/100

Epoch 1: val_loss improved from None to 0.41403, saving model to best_model.h5



Epoch 1: finished saving model to best_model.h5
210/210 - 3s - 14ms/step - accuracy: 0.7807 - loss: 0.4885 - val_accuracy: 0.8121 - val_loss: 0.4140 - learning_rate: 0.0010
Epoch 2/100

Epoch 2: val_loss improved from 0.41403 to 0.39167, saving model to best_model.h5



Epoch 2: finished saving model to best_model.h5
210/210 - 1s - 3ms/step - accuracy: 0.8146 - loss: 0.4313 - val_accuracy: 0.8230 - val_loss: 0.3917 - learning_rate: 0.0010
Epoch 3/100

Epoch 3: val_loss improved from 0.39167 to 0.37103, saving model to best_model.h5



Epoch 3: finished saving model to best_model.h5
210/210 - 1s - 3ms/step - accuracy: 0.8261 - loss: 0.4154 - val_accuracy: 0.8461 - val_loss: 0.3710 - learning_rate: 0.0010
Epoch 4/100

Epoch 4: val_loss improved from 0.37103 to 0.34867, saving model to best_model.h5



Epoch 4: finished saving model to best_model.h5
210/210 - 1s - 4ms/step - accuracy: 0.8355 - loss: 0.3939 - val_accuracy: 0.8679 - val_loss: 0.3487 - learning_rate: 0.0010
Epoch 5/100

Epoch 5: val_loss improved from 0.34867 to 0.34184, saving model to best_model.h5


...


Epoch 7: finished saving model to best_model.h5
210/210 - 1s - 4ms/step - accuracy: 0.8454 - loss: 0.3650 - val_accuracy: 0.8709 - val_loss: 0.3330 - learning_rate: 0.0010
Epoch 8/100

Epoch 8: val_loss improved from 0.33299 to 0.33049, saving model to best_model.h5



Epoch 8: finished saving model to best_model.h5
210/210 - 1s - 3ms/step - accuracy: 0.8525 - loss: 0.3601 - val_accuracy: 0.8715 - val_loss: 0.3305 - learning_rate: 0.0010
Epoch 9/100

Epoch 9: val_loss did not improve from 0.33049
210/210 - 1s - 4ms/step - accuracy: 0.8500 - loss: 0.3535 - val_accuracy: 0.8745 - val_loss: 0.3321 - learning_rate: 0.0010
Epoch 10/100

Epoch 10: val_loss did not improve from 0.33049
210/210 - 1s - 4ms/step - accuracy: 0.8522 - loss: 0.3516 - val_accuracy: 0.8752 - val_loss: 0.3314 - learning_rate: 0.0010
Epoch 11/100

Epoch 11: val_loss did not improve from 0.33049
210/210 - 1s - 3ms/step - accuracy: 0.8555 - loss: 0.3534 - val_accuracy: 0.8721 - val_loss: 0.3329 - learning_rate: 0.0010
Epoch 12/100

Epoch 12: val_loss did not improve from 0.33049
210/210 - 1s - 4ms/step - accuracy: 0.8540 - loss: 0.3507 - val_accuracy: 0.8703 - val_loss: 0.3325 - learning_rate: 0.0010
Epoch 13/100

Epoch 13: val_loss improved from 0.33049 to 0.32837, saving model to be


Epoch 13: finished saving model to best_model.h5
210/210 - 1s - 4ms/step - accuracy: 0.8572 - loss: 0.3488 - val_accuracy: 0.8812 - val_loss: 0.3284 - learning_rate: 0.0010
Epoch 14/100

Epoch 14: val_loss did not improve from 0.32837
210/210 - 1s - 4ms/step - accuracy: 0.8551 - loss: 0.3465 - val_accuracy: 0.8752 - val_loss: 0.3295 - learning_rate: 0.0010
Epoch 15/100

Epoch 15: val_loss did not improve from 0.32837
210/210 - 1s - 4ms/step - accuracy: 0.8569 - loss: 0.3455 - val_accuracy: 0.8709 - val_loss: 0.3301 - learning_rate: 0.0010
Epoch 16/100

Epoch 16: val_loss did not improve from 0.32837
210/210 - 1s - 4ms/step - accuracy: 0.8593 - loss: 0.3441 - val_accuracy: 0.8733 - val_loss: 0.3310 - learning_rate: 0.0010
Epoch 17/100

Epoch 17: val_loss did not improve from 0.32837
210/210 - 1s - 4ms/step - accuracy: 0.8600 - loss: 0.3425 - val_accuracy: 0.8721 - val_loss: 0.3293 - learning_rate: 0.0010
Epoch 18/100

Epoch 18: val_loss did not improve from 0.32837

Epoch 18: ReduceLRO


Epoch 20: finished saving model to best_model.h5
210/210 - 1s - 4ms/step - accuracy: 0.8616 - loss: 0.3400 - val_accuracy: 0.8758 - val_loss: 0.3282 - learning_rate: 5.0000e-04
Epoch 21/100

Epoch 21: val_loss did not improve from 0.32822
210/210 - 1s - 3ms/step - accuracy: 0.8606 - loss: 0.3381 - val_accuracy: 0.8758 - val_loss: 0.3284 - learning_rate: 5.0000e-04
Epoch 22/100

Epoch 22: val_loss improved from 0.32822 to 0.32714, saving model to best_model.h5



Epoch 22: finished saving model to best_model.h5
210/210 - 1s - 4ms/step - accuracy: 0.8631 - loss: 0.3367 - val_accuracy: 0.8764 - val_loss: 0.3271 - learning_rate: 5.0000e-04
Epoch 23/100

Epoch 23: val_loss did not improve from 0.32714
210/210 - 1s - 3ms/step - accuracy: 0.8594 - loss: 0.3376 - val_accuracy: 0.8739 - val_loss: 0.3280 - learning_rate: 5.0000e-04
Epoch 24/100

Epoch 24: val_loss did not improve from 0.32714
210/210 - 1s - 3ms/step - accuracy: 0.8633 - loss: 0.3354 - val_accuracy: 0.8733 - val_loss: 0.3282 - learning_rate: 5.0000e-04
Epoch 25/100

Epoch 25: val_loss did not improve from 0.32714
210/210 - 1s - 4ms/step - accuracy: 0.8615 - loss: 0.3342 - val_accuracy: 0.8733 - val_loss: 0.3286 - learning_rate: 5.0000e-04
Epoch 26/100

Epoch 26: val_loss did not improve from 0.32714
210/210 - 1s - 3ms/step - accuracy: 0.8579 - loss: 0.3394 - val_accuracy: 0.8721 - val_loss: 0.3313 - learning_rate: 5.0000e-04
Epoch 27/100

Epoch 27: val_loss did not improve from 0.32714


In [23]:
# 6) Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss, "Test acc:", test_acc)

Test loss: 0.3367924988269806 Test acc: 0.8569697141647339


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(epochs, acc, marker='o', label='train')
plt.plot(epochs, val_acc, marker='s', label='val')
plt.title('Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True)
plt.ylim(0.0, 1.0)

plt.subplot(1,2,2)
plt.plot(epochs, loss, marker='o', label='train')
plt.plot(epochs, val_loss, marker='s', label='val')
plt.title('Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
miny = min(min(loss), min(val_loss)); maxy = max(max(loss), max(val_loss))
pad = max((maxy-miny)*0.1, 1e-6)
plt.ylim(max(0, miny-pad), maxy+pad)

plt.tight_layout(); plt.show()